[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lowdanie/hartree-fock-solver/blob/main/notebooks/geometry_optimization.ipynb)

# Molecular Geometry Optimization

This notebook demonstrates differentiable quantum chemistry using
[slaterform](https://github.com/lowdanie/hartree-fock-solver).

We optimize the geometry of various molecules by differentiating through `slaterform`'s Hartree-Fock SCF solver using `jax`.

An animation of the electronic density trajectory is generated at the end.

## System Setup

Run the following cells to initialize the optimization loop controller, visualization utilities and molecule definitions.

In [ ]:
# @title Pip Installs

!pip install -qq py3Dmol
!pip install -qq git+https://github.com/lowdanie/hartree-fock-solver


In [ ]:
!pip freeze | grep slaterform

In [ ]:
# @title Imports { display-mode: "form" }

import jax

jax.config.update("jax_enable_x64", True)

import dataclasses
import io
import time
from typing import Callable, NamedTuple
from collections.abc import Sequence
import functools

import jax.numpy as jnp
import numpy as np
import optax

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

import pubchempy as pcp
import py3Dmol

import slaterform as sf
import slaterform.hartree_fock.scf as scf

In [ ]:
print(f"JAX Backend: {jax.devices()[0]}")

In [ ]:
# @title Optimization Loop { display-mode: "form" }


class OptimizerState(NamedTuple):
    positions: jax.Array
    opt_state: optax.OptState
    density: jax.Array


class ElectronicState(NamedTuple):
    density: jax.Array


class Snapshot(NamedTuple):
    """Snapshot of one optimization step."""

    positions: jax.Array
    energy: jax.Array
    density: jax.Array


def init_optimizer(
    optimizer: optax.GradientTransformation,
    molecule: sf.Molecule,
) -> OptimizerState:
    positions = jnp.array([a.position for a in molecule.atoms])
    opt_state = optimizer.init(positions)
    density = scf.build_initial_density(molecule)

    return OptimizerState(positions, opt_state, density)


def build_optimization_step(
    energy_and_grad_fn: Callable,
    optimizer: optax.GradientTransformation,
    basis: sf.BatchedBasis,
    fixed_atomic_indices: Sequence[int] = [],
) -> Callable:
    mask = jnp.ones((len(basis.atoms), 1))
    if fixed_atomic_indices:
        mask = mask.at[jnp.array(fixed_atomic_indices)].set(0.0)

    def step(carry: OptimizerState, _) -> tuple[OptimizerState, Snapshot]:
        positions, opt_state, density = carry
        (E, electronic_state), grad = energy_and_grad_fn(
            positions, basis, density
        )

        updates, next_opt_state = optimizer.update(
            grad * mask, opt_state, positions
        )
        next_positions = optax.apply_updates(positions, updates)

        next_density = jax.lax.stop_gradient(electronic_state.density)
        next_carry = OptimizerState(
            next_positions, next_opt_state, next_density
        )

        snapshot = Snapshot(positions=positions, energy=E, density=next_density)

        return next_carry, snapshot

    return step


def run_optimization(
    step_fn: Callable, optimizer_state: OptimizerState, n_steps: int
) -> tuple[OptimizerState, Snapshot]:
    return jax.lax.scan(step_fn, optimizer_state, length=n_steps)

In [ ]:
# @title Optimization Dashboard { display-mode: "form" }


def _get_lims(values, min_span=0.1, padding_fraction=0.05):
    if not values:
        return 0, 1

    min_val = min(values)
    max_val = max(values)
    span = max_val - min_val

    if span < min_span:
        mid = (max_val + min_val) / 2.0
        return mid - (min_span / 2.0), mid + (min_span / 2.0)

    pad = span * padding_fraction
    return min_val - pad, max_val + pad


class Dashboard:
    def __init__(self, total_steps, min_y_span=0.1):
        self.total_steps = total_steps
        self.min_y_span = min_y_span
        self.energies = []
        self.steps = []

        color = "tab:blue"

        self.fig, self.ax = plt.subplots(figsize=(8, 4))
        self.ax.set_title("Optimization Progress")
        self.ax.set_xlabel("Step")
        self.ax.set_ylabel("Total Energy (Hartree)", color=color)
        self.ax.grid(True, alpha=0.3, linestyle="--")
        self.ax.tick_params(axis="y", labelcolor=color)

        formatter = ticker.ScalarFormatter(useOffset=False)
        formatter.set_scientific(False)
        self.ax.yaxis.set_major_formatter(formatter)

        (self.line,) = self.ax.plot([], [], color=color, lw=2, label="Energy")
        self.ax.legend(loc="upper right")

        plt.close(self.fig)
        self.display_handle = None

    def update(self, snapshot: Snapshot):
        self.energies.extend(snapshot.energy)
        self.steps = range(len(self.energies))

        if self.display_handle is None:
            self.display_handle = display(self.fig, display_id=True)

        self.line.set_data(self.steps, self.energies)

        ymin, ymax = _get_lims(self.energies, self.min_y_span)
        self.ax.set_ylim(ymin, ymax)
        self.ax.set_xlim(0, max(self.total_steps, len(self.steps)))

        self.display_handle.update(self.fig)

In [ ]:
# @title Molecule Renderer  { display-mode: "form" }


@dataclasses.dataclass
class Frame:
    step: int
    energy: float
    density_data: str


def _build_frames_from_snapshot(
    snapshot: Snapshot,
    first_step: int,
    template_mol: sf.Molecule,
    template_basis: sf.BatchedBasis,
    grid: sf.RegularGrid,
) -> list[Frame]:
    frames = []

    for i in range(len(snapshot.energy)):
        step = first_step + i
        energy = snapshot.energy[i]
        positions = snapshot.positions[i]
        mol = template_mol.with_positions(positions)
        basis = template_basis.with_positions(positions)

        rho = sf.analysis.evaluate_density(
            basis.basis_blocks,
            snapshot.density[i],
            grid,
        )

        with io.StringIO() as buffer:
            sf.analysis.write_cube_data(
                mol=mol,
                grid=grid,
                data=rho,
                description=f"Step {step}",
                f=buffer,
            )
            frames.append(Frame(step, energy, buffer.getvalue()))

    return frames


def build_frames(
    template_mol: sf.Molecule,
    template_basis: sf.BatchedBasis,
    history: Sequence[Snapshot],
    resolution=25,
) -> Sequence[Frame]:
    grid = sf.analysis.build_bounding_grid(
        template_mol, padding=3.0, spacing=10.0 / resolution
    )
    frames = []

    for snapshot in history:
        frames.extend(
            _build_frames_from_snapshot(
                snapshot, len(frames), template_mol, template_basis, grid
            )
        )

    return frames


def _render(frame, view):
    view.removeAllModels()
    view.removeAllShapes()
    view.removeAllLabels()

    view.addModel(frame.density_data, "cube")
    view.setStyle({"stick": {"radius": 0.15}, "sphere": {"scale": 0.3}})

    view.addVolumetricData(
        frame.density_data,
        "cube",
        {
            "algo": "volume",
            "transferfn": [
                {"value": 0.00, "color": "white", "opacity": 0.0},
                {"value": 0.005, "color": "blue", "opacity": 0.002},
                {"value": 0.05, "color": "blue", "opacity": 0.01},
                {"value": 0.20, "color": "blue", "opacity": 0.05},
            ],
            "smoothness": 5,
        },
    )


def _update_status(frame, status_label):
    energy = frame.energy
    status_label.value = (
        f"<div style='font-family: monospace; font-size: 14px;'>"
        f"<b>Energy:</b> {energy:.6f} Ha"
        f"</div>"
    )


def molecule_viewer_app(frames):
    display(
        HTML(
            '<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'
        )
    )
    slider = widgets.IntSlider(
        min=0, max=len(frames) - 1, step=1, description="Step:"
    )
    play = widgets.Play(
        value=0,
        min=0,
        max=len(frames) - 1,
        step=1,
        interval=2000,
        description="Play",
        show_repeat=False,
    )
    widgets.jslink((play, "value"), (slider, "value"))

    status_label = widgets.HTML(
        value="<b>Initializing...</b>",
        layout=widgets.Layout(margin="0 0 0 20px", padding="5px"),
    )
    view = py3Dmol.view(width=700, height=500)
    output_container = widgets.Output()

    def on_change(change):
        if change["name"] == "value":
            frame = frames[change["new"]]
            _render(frame, view)
            _update_status(frame, status_label)
            view.update()

    slider.observe(on_change)

    with output_container:
        _render(frames[0], view)
        _update_status(frames[0], status_label)
        view.show()
        view.zoomTo()

    controls = widgets.HBox([play, slider, status_label])
    layout = widgets.VBox([controls, output_container])

    return layout

In [ ]:
# @title Molecule Zoo{ display-mode: "form" }


@dataclasses.dataclass
class ExperimentConfig:
    """Configuration for a single experiment."""

    name: str
    molecule: sf.Molecule
    fixed_indices: list[int]


def build_water():
    oh_dist = 1.8
    mol = sf.Molecule.from_geometry(
        [
            sf.Atom("O", 8, jnp.array([0.0, 0.0, 0.0])),
            sf.Atom("H", 1, jnp.array([-oh_dist, 0.1, 0.0])),
            sf.Atom("H", 1, jnp.array([oh_dist, 0.1, 0.0])),
        ],
        basis_name="sto-3g",
    )

    return mol, [0]  # fix oxygen


def build_methane():
    ch_dist = 2.0

    mol = sf.Molecule.from_geometry(
        [
            sf.Atom("C", 6, jnp.array([0.0, 0.0, 0.0])),
            sf.Atom("H", 1, jnp.array([ch_dist, 0.0, 0.1])),
            sf.Atom("H", 1, jnp.array([-ch_dist, 0.0, 0.1])),
            sf.Atom("H", 1, jnp.array([0.0, ch_dist, -0.1])),
            sf.Atom("H", 1, jnp.array([0.0, -ch_dist, 0.1])),
        ],
        basis_name="sto-3g",
    )
    return mol, [0]  # fix carbon


def build_ammonia():
    nh_dist = 1.9
    nh_x = nh_dist * np.sin(np.pi / 6)
    nh_y = nh_dist * np.cos(np.pi / 6)
    mol = sf.Molecule.from_geometry(
        [
            sf.Atom("N", 7, jnp.array([0.0, 0.0, 0.0])),
            sf.Atom("H", 1, jnp.array([nh_dist, 0.0, 0.1])),
            sf.Atom("H", 1, jnp.array([-nh_x, nh_y, 0.1])),
            sf.Atom("H", 1, jnp.array([-nh_x, -nh_y, -0.1])),
        ],
        basis_name="sto-3g",
    )
    return mol, [0]  # fix nitrogen


def build_ethylene():
    # Start twisted 90 degrees (Broken Pi-bond)
    # The hydrogens on the left are flat (XY plane)
    # The hydrogens on the right are vertical (XZ plane)
    cc_dist = 2.5
    ch_dist = 2.0
    h_delta = ch_dist / np.sqrt(2)

    mol = sf.Molecule.from_geometry(
        [
            sf.Atom("C", 6, jnp.array([0.0, 0.0, 0.0])),
            sf.Atom("C", 6, jnp.array([cc_dist, 0.0, 0.0])),
            # Left Hydrogens (Flat)
            sf.Atom("H", 1, jnp.array([-h_delta, h_delta, 0.0])),
            sf.Atom("H", 1, jnp.array([-h_delta, -h_delta, 0.0])),
            # Right Hydrogens (Twisted 90 degrees up/down)
            sf.Atom("H", 1, jnp.array([cc_dist + h_delta, 0.0, h_delta])),
            sf.Atom("H", 1, jnp.array([cc_dist + h_delta, 0.0, -h_delta])),
        ],
        basis_name="sto-3g",
    )

    # Fix one carbon
    return mol, [0]


def build_aspirin():
    compounds = pcp.get_compounds("Aspirin", "name", record_type="3d")
    atoms = sf.adapters.pubchem.load_geometry(compounds[0])
    mol = sf.Molecule.from_geometry(atoms, basis_name="sto-3g")

    return mol, []


def build_benzene():
    compounds = pcp.get_compounds("Benzene", "name", record_type="3d")
    atoms = sf.adapters.pubchem.load_geometry(compounds[0])
    mol = sf.Molecule.from_geometry(atoms, basis_name="sto-3g")

    # Perturb in the z direction.
    pos = np.array([a.position for a in mol.atoms])
    pos[::2, 2] += 0.5
    pos[1::2, 2] -= 0.5

    rng = np.random.default_rng(123)
    noise = rng.normal(scale=0.1, size=pos.shape)
    pos += noise

    mol = mol.with_positions(pos)

    return mol, []


def build_butane():
    atoms = [sf.Atom("C", 6, jnp.zeros(3)) for _ in range(4)] + [
        sf.Atom("H", 1, jnp.zeros(3)) for _ in range(10)
    ]
    mol = sf.Molecule.from_geometry(atoms, basis_name="sto-3g")

    pos = np.zeros((len(mol.atoms), 3))
    cc_dist = 2.5
    ch_dist = 2.0

    # Place 4 Carbons along the X-axis.
    c_x = np.linspace(-1.5 * cc_dist, 1.5 * cc_dist, 4)
    pos[:4, 0] = c_x

    # Place 4 Hydrogens above the Carbon chain.
    pos[4:8, 0] = c_x
    pos[4:8, 1] = ch_dist
    pos[4:8, 2] = np.array([1.5, -1.5, 1.5, -1.5])

    # Place 4 Hydrogens below.
    pos[8:12, 0] = c_x
    pos[8:12, 1] = -ch_dist
    pos[8:12, 2] = np.array([1.5, -1.5, 1.5, -1.5])

    # Place one on each end.
    pos[12, 0] = c_x[0] - ch_dist
    pos[12, 2] = 1.0

    pos[13, 0] = c_x[-1] + ch_dist
    pos[13, 2] = -1.0

    rng = np.random.default_rng(123)
    pos += rng.normal(scale=0.05, size=pos.shape)

    mol = mol.with_positions(pos)
    return mol, []


EXPERIMENT_FACTORIES = {
    "Water (H2O)": build_water,
    "Methane (CH4)": build_methane,
    "Ammonia (NH4)": build_ammonia,
    "Ethylene (C2H4)": build_ethylene,
    "Aspirin (C9H8O4)": build_aspirin,
    "Benzene (C6H6)": build_benzene,
    "Butane (C4H10)": build_butane,
}


def load_experiment(name: str) -> ExperimentConfig:
    builder_func = EXPERIMENT_FACTORIES[name]
    mol, fixed_indices = builder_func()
    return ExperimentConfig(name, mol, fixed_indices)

# Differentiable Energy

Use `slaterform`'s Hartree-Fock SCF solver to define a differentiable molecular energy function.

In [ ]:
def total_energy(
    positions: jax.Array,
    template_basis: sf.BatchedBasis,
    P0: jax.Array,
    options: scf.Options,
):
    """Total energy of the molecule with the specified atomic positions.

    The template molecule determines the atomic numbers and basis set.
    """
    basis = template_basis.with_positions(positions)
    result = scf.solve(basis, options, P0)

    return (
        result.total_energy,
        ElectronicState(result.density),
    )


options = scf.Options(
    solver=sf.fixed_point.AndersonParams(
        max_iter=50, tol=1e-7, m=5, beta=0.7, static_loop=False
    ),
    integral_strategy=scf.CachedStrategy(dtype=jnp.float32),
    perturbation=1e-10,
    implicit_diff=True,
)

energy_fn = functools.partial(total_energy, options=options)
total_energy_and_grad = jax.value_and_grad(energy_fn, has_aux=True)

# Run Simulation

Select a molecule below and watch the geometry evolve to minimize the energy.

In [ ]:
# @title Select Molecule { display-mode: "form" }
# @markdown Choose a simulation scenario from the dropdown.

experiment_name = "Butane (C4H10)"  # @param ["Water (H2O)", "Methane (CH4)", "Ammonia (NH4)", "Ethylene (C2H4)",  "Aspirin (C9H8O4)", "Benzene (C6H6)", "Butane (C4H10)"]
experiment_cfg = load_experiment(experiment_name)

print(f"✅ Loaded: {experiment_name}")
print(f"   • Atoms: {[atom.symbol for atom in experiment_cfg.molecule.atoms]}")
print(f"   • Fixed Atoms Indices: {experiment_cfg.fixed_indices}")

In [ ]:
# @title Configure The Optimizer

n_chunks = 5
chunk_size = 10
n_steps = n_chunks * chunk_size

scheduler = optax.cosine_decay_schedule(
    init_value=0.05, decay_steps=100, alpha=0.05
)
optimizer = optax.adam(learning_rate=scheduler)
optimizer_state = init_optimizer(optimizer, experiment_cfg.molecule)

template_basis = sf.BatchedBasis.from_molecule(
    experiment_cfg.molecule, batch_size_1e=64, batch_size_2e=256
)
step_fn = build_optimization_step(
    total_energy_and_grad,
    optimizer,
    template_basis,
    experiment_cfg.fixed_indices,
)


@jax.jit
def run_chunk(state: OptimizerState) -> tuple[OptimizerState, Snapshot]:
    return run_optimization(step_fn, state, n_steps=chunk_size)


optimizer_state = init_optimizer(optimizer, experiment_cfg.molecule)
history = []

In [ ]:
print("Warming up the JAX kernel...")
optimizer_state, snapshot = run_chunk(optimizer_state)
history.append(snapshot)

In [ ]:
dashboard = Dashboard(n_steps)
dashboard.update(history[-1])

for i in range(n_chunks - 1):
    optimizer_state, snapshot = run_chunk(optimizer_state)
    history.append(snapshot)
    dashboard.update(snapshot)

In [ ]:
# @title Electronic State Trajectory  { display-mode: "form" }

resolution = 20
frames = build_frames(
    experiment_cfg.molecule, template_basis, history, resolution
)
clear_output(wait=True)
display(molecule_viewer_app(frames))